# <span style="color:green"> Script for processing excel files to calculate average, std, min, max, and 5%-95% of data
   

# Load libraries

In [1]:
import os
import pandas as pd
import re
import openpyxl
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from pathlib import Path


# Process sheets in excel

### PaperMill parameters to be changed

In [2]:
# Input file
file_path = 'Tail1/T1_CEP1_M.xlsx'

# Output file
output_file = 'T1_ALL.xlsx'


### Remove useless columns from sheet "radii"

In [ ]:
# Load the Excel file and the "radii" sheet
df = pd.read_excel(file_path, sheet_name='radii')

# Drop specific columns by name
columns_to_remove = ['node_id', 'parent_id', 'x', 'y', 'z']
existing_cols = [col for col in columns_to_remove if col in df.columns]

if existing_cols:
    df = df.drop(columns=existing_cols)
else:
    print("No columns to remove found in the DataFrame.")

# Save back to the same file or a new one
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name='radii', index=False)

### Remove useless columns from sheet "paths_n"

In [3]:
# Load all sheet names
xls = pd.ExcelFile(file_path)
all_sheets = xls.sheet_names

# Filter sheets matching the pattern "path_<number>"
target_sheets = [name for name in all_sheets if re.fullmatch(r'paths_\d+', name)]

# Process and modify those sheets
updated_sheets = {}
for sheet in target_sheets:
    df = pd.read_excel(file_path, sheet_name=sheet)
    df = df.drop(columns=['Index', 'Path'], errors='ignore')
    updated_sheets[sheet] = df

# Write back the modified sheets, replacing them
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    for sheet_name, df in updated_sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)


### Bring all pahts into one sheet

In [4]:
# Filter out "paths_1" from the list, keep only sheets after it
sheets_to_concat = [sheet for sheet in target_sheets if sheet != 'paths_1']

if not sheets_to_concat:
    print("Only 'paths_1' exists, nothing to concatenate.")
else:
    # Read existing data from 'paths_1'
    existing_df = pd.read_excel(file_path, sheet_name='paths_1')

    # Collect 'Distance' columns from sheets after 'paths_1'
    distances_list = []
    for i, sheet in enumerate(sheets_to_concat, start=2):
        df = pd.read_excel(file_path, sheet_name=sheet)
        if 'Distance' in df.columns:
            renamed_df = df[['Distance']].rename(columns={'Distance': f'Distance_{i}'})
            distances_list.append(renamed_df)
        else:
            print(f"Sheet '{sheet}' has no 'Distance' column.")


    if distances_list:
        combined_distances = pd.concat(distances_list, axis=1)
    else:
        combined_distances = pd.DataFrame()

    # Concatenate existing paths_1 data with combined Distance columns
    result_df = pd.concat([existing_df, combined_distances], axis=1)

    # Write updated data back to 'paths_1', replacing that sheet only
    with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        result_df.to_excel(writer, sheet_name='paths_1', index=False)


Only 'paths_1' exists, nothing to concatenate.


In [5]:
wb = openpyxl.load_workbook(file_path)
for sheet_name in sheets_to_concat:
    if sheet_name in wb.sheetnames:
        std = wb[sheet_name]
        wb.remove(std)
wb.save(file_path)
wb.close()                # Ensure it’s closed


### Get statistics of Radii

In [6]:
# Load data
df = pd.read_excel(file_path, sheet_name='radii')

# Compute full stats
percentile_5 = df['radius'].quantile(0.05)
percentile_95 = df['radius'].quantile(0.95)

full_stats = {
    'mean': df['radius'].mean(),
    'std.P': df['radius'].std(ddof=0),
    'min': df['radius'].min(),
    'max': df['radius'].max(),
    'p5': percentile_5,
    'p95': percentile_95
}

# Filter data between 5th and 95th percentiles
filtered = df[(df['radius'] >= percentile_5) & (df['radius'] <= percentile_95)]

# Compute filtered stats
filtered_stats = {
    'mean': filtered['radius'].mean(),
    'std.P': filtered['radius'].std(ddof=0),
    'min': filtered['radius'].min(),
    'max': filtered['radius'].max(),
    'p5': percentile_5,
    'p95': percentile_95
}

# Open workbook and target sheet
wb = load_workbook(file_path)
ws = wb['radii']

# Determine starting column for writing stats (leave one column for row titles)
start_col = ws.max_column + 3  # +3 to leave one empty column between data and stats

# Write headers with a leading empty cell for row titles
ws.cell(row=1, column=start_col - 1, value="")  # header for row titles
for i, key in enumerate(full_stats.keys()):
    ws.cell(row=1, column=start_col + i, value=key)

# Write "Full Data" row title and values
ws.cell(row=2, column=start_col - 1, value="Full Data")
for i, value in enumerate(full_stats.values()):
    ws.cell(row=2, column=start_col + i, value=value)

# Write "Filtered Data" row title and values
ws.cell(row=3, column=start_col - 1, value="Filtered Data")
for i, value in enumerate(filtered_stats.values()):
    ws.cell(row=3, column=start_col + i, value=value)

# Save workbook
wb.save(file_path)
wb.close()                # Ensure it’s closed


### Get statistics of Paths

In [7]:
# Load the Excel file and the 'paths_1' sheet
df = pd.read_excel(file_path, sheet_name='paths_1')

# Extract all columns named 'Distance' (even if duplicated)
distance_columns = [col for col in df.columns if 'Distance' in col]

if not distance_columns:
    raise ValueError("No 'Distance' columns found in 'paths_1'.")

# Combine all Distance columns into one long Series (flatten)
all_distances = pd.concat([df[col] for col in distance_columns], ignore_index=True).dropna()

# Compute statistics on all distances
p5 = all_distances.quantile(0.05)
p95 = all_distances.quantile(0.95)

full_stats = {
    'mean': all_distances.mean(),
    'std.P': all_distances.std(ddof=0),
    'min': all_distances.min(),
    'max': all_distances.max(),
    'p5': p5,
    'p95': p95
}

# Filter data between 5th and 95th percentiles
filtered_distances = all_distances[(all_distances >= p5) & (all_distances <= p95)]

filtered_stats = {
    'mean': filtered_distances.mean(),
    'std.P': filtered_distances.std(ddof=0),
    'min': filtered_distances.min(),
    'max': filtered_distances.max(),
    'p5': p5,
    'p95': p95
}

# Write stats to the right side of the 'paths_1' sheet
wb = load_workbook(file_path)
ws = wb['paths_1']

# Find first empty column to the right of existing data
start_col = ws.max_column + 3  # Leave one column space

# Write header row
ws.cell(row=1, column=start_col - 1, value="")  # For row titles
for i, key in enumerate(full_stats.keys()):
    ws.cell(row=1, column=start_col + i, value=key)

# Write full stats
ws.cell(row=2, column=start_col - 1, value="Full Data")
for i, value in enumerate(full_stats.values()):
    ws.cell(row=2, column=start_col + i, value=value)

# Write filtered stats
ws.cell(row=3, column=start_col - 1, value="Filtered Data")
for i, value in enumerate(filtered_stats.values()):
    ws.cell(row=3, column=start_col + i, value=value)

# Save the updated Excel file
wb.save(file_path)
wb.close()                # Ensure it’s closed


### Add results to T1_ALL

In [23]:
import os
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows

# Extract source file name (without path and extension)
source_name = os.path.splitext(os.path.basename(file_path))[0]

# Load statistics from the original file
wb = load_workbook(file_path)
ws_radii = wb['radii']
ws_paths = wb['paths_1']

# Extract stats from 'radii' sheet (assumes they were written starting after the data)
radii_data = []
for row in ws_radii.iter_rows(min_row=2, max_row=3, values_only=True):
    radii_data.append([source_name] + list(row[-6:]))  # Skip title cell in column 1

radii_df = pd.DataFrame(radii_data, columns=['File', 'mean', 'std.P', 'min', 'max', 'p5', 'p95'])
radii_df.insert(1, 'Type', ['Full Data', 'Filtered Data'])

# Extract stats from 'paths_1' sheet
paths_data = []
for row in ws_paths.iter_rows(min_row=2, max_row=3, values_only=True):
    paths_data.append([source_name] + list(row[-6:]))

paths_df = pd.DataFrame(paths_data, columns=['File', 'mean', 'std.P', 'min', 'max', 'p5', 'p95'])
paths_df.insert(1, 'Type', ['Full Data', 'Filtered Data'])

# Prepare output file and write it
output_path = Path(file_path).parent / output_file

# Function to append to sheet or create new one
def append_df_to_excel(df, sheet_name):
    if os.path.exists(output_path):
        book = load_workbook(output_path)
        if sheet_name in book.sheetnames:
            sheet = book[sheet_name]
            start_row = sheet.max_row + 1
            for r in dataframe_to_rows(df, index=False, header=False):
                sheet.append(r)
            book.save(output_path)  # Save after appending!
        else:
            with pd.ExcelWriter(output_path, engine='openpyxl', mode='a') as writer:
                df.to_excel(writer, sheet_name=sheet_name, index=False)
    else:
        with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
            df.to_excel(writer, sheet_name=sheet_name, index=False)


# Append data to the appropriate sheets
append_df_to_excel(radii_df, sheet_name='radii')
append_df_to_excel(paths_df, sheet_name='Paths')
